# Student Information

**Name**: ________________________________

**Roll Number**: ________________________________

**Date**: ________________________________

**Batch**: ________________________________

---



# Unit 3, Lecture 5: Euler Angles & 3D Rotations

**Course**: Dynamics of Systems and Machines (DSM)  
**Instructor**: Sunny  
**Duration**: 1 hour  
**Unit**: Rigid Body Kinematics

---

## Learning Objectives

By the end of this lecture, you will be able to:

1. **Describe** 3D rigid body orientation using Euler angles
2. **Construct** Direction Cosine Matrices (DCM) for rotation sequences
3. **Calculate** angular velocity from Euler angle rates
4. **Apply** 3D kinematics to mechanism analysis (slider-crank)
5. **Integrate** all concepts from Unit 3 into comprehensive problems
6. **Prepare** for rigid body kinetics (Unit 5)

---

## Unit 3 Journey: Where We've Been

### Lectures 1-4 Recap

**L1: Fixed Frame Kinematics**
- Velocity: v = v_center + ω × r
- 2D rigid body motion

**L2: Fixed Frame Acceleration**
- Acceleration: a = a_center + α × r + ω × (ω × r)
- Tangential and normal components

**L3: Transport Theorem**
- Relating fixed and rotating frames
- v_fixed = v_rel + Ω × r

**L4: Coriolis Acceleration**
- a_fixed = a_rel + 2Ω × v_rel + α × r + Ω × (Ω × r)
- Real-world applications

### Today: The Final Piece

**Question**: How do we describe 3D orientations systematically?

**Answer**: **Euler angles** - a powerful way to parameterize rotations!

**Goal**: Complete the kinematics toolkit for rigid bodies in 3D space.

---

## Why Euler Angles Matter

### The 3D Orientation Challenge

**2D rotation**: Single angle θ describes orientation completely

**3D rotation**: Need **THREE** angles to describe orientation
- Which three?
- In what order?
- How to compute angular velocity?

**Euler angles provide systematic answers!**

### Real-World Applications

1. **Aircraft dynamics**: Roll, pitch, yaw
2. **Spacecraft attitude**: 3-axis orientation control
3. **Robotics**: 6-DOF manipulators
4. **Mechanism analysis**: Slider-crank, universal joints
5. **Computer graphics**: 3D animations, games
6. **IMU sensors**: Inertial measurement units

### Course Integration

This lecture **bridges**:
- **Unit 3** (Kinematics): How things move
- → **Unit 5** (Kinetics): Why things move (forces/torques)

Euler angles + angular velocity → Ready for equations of motion!

---

## Required Libraries

SymPy for symbolic DCM calculations, animations for mechanisms.

---

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML
import sympy as sp
from sympy import symbols, cos, sin, Matrix, simplify, latex
from scipy.integrate import odeint

# Custom animation library
import sys
sys.path.append('../src/animations')
from rotating_frames import animate_slider_crank

# Plot styling
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11
plt.rcParams['lines.linewidth'] = 2

print("All libraries loaded successfully!")
print("Ready for 3D rotations!")
print()


## Euler Angles: Parameterizing 3D Rotations

### The Problem

**3D orientation**: A rigid body in 3D space has **3 rotational DOF**

**Question**: How to describe orientation with 3 parameters?

**Answer**: **Euler angles** - sequence of three rotations about coordinate axes

### Many Conventions!

**Warning**: There are **12 different Euler angle conventions**!
- 6 "proper" Euler sequences: XYX, XZX, YXY, YZY, ZXZ, ZYZ
- 6 "Tait-Bryan" sequences: XYZ, XZY, YXZ, YZX, ZXY, ZYX

**Most common in engineering**:
- **Aerospace**: 3-2-1 sequence (ZYX) → Roll, Pitch, Yaw
- **Robotics**: Various, depends on application
- **This course**: We'll use **3-1-3 sequence (ZXZ)** - classic Euler angles

**Key point**: Always specify your convention!

---

## Basic Rotation Matrices

### Single-Axis Rotations

**Rotation about X-axis by angle α**:
$$R_x(\alpha) = \begin{bmatrix}
1 & 0 & 0 \\
0 & \cos\alpha & -\sin\alpha \\
0 & \sin\alpha & \cos\alpha
\end{bmatrix}$$

**Rotation about Y-axis by angle β**:
$$R_y(\beta) = \begin{bmatrix}
\cos\beta & 0 & \sin\beta \\
0 & 1 & 0 \\
-\sin\beta & 0 & \cos\beta
\end{bmatrix}$$

**Rotation about Z-axis by angle γ**:
$$R_z(\gamma) = \begin{bmatrix}
\cos\gamma & -\sin\gamma & 0 \\
\sin\gamma & \cos\gamma & 0 \\
0 & 0 & 1
\end{bmatrix}$$

**Properties**:
- Orthogonal: $R^T R = I$
- Determinant: $\det(R) = 1$
- Inverse: $R^{-1} = R^T$

---

## Euler Angles: 3-1-3 Sequence (ZXZ)

### The Classic Euler Sequence

**Sequence**: Rotate about **Z**, then **X'**, then **Z''**

**Three angles**:
1. **φ** (phi): First rotation about Z-axis (precession)
2. **θ** (theta): Second rotation about new X-axis (nutation)
3. **ψ** (psi): Third rotation about newest Z-axis (spin)

### Direction Cosine Matrix (DCM)

**Combine rotations** (order matters!):
$$R = R_z(\phi) \cdot R_x(\theta) \cdot R_z(\psi)$$

**Expanded form**:
$$R_{ZXZ}(\phi, \theta, \psi) = \begin{bmatrix}
c\phi c\psi - s\phi c\theta s\psi & -c\phi s\psi - s\phi c\theta c\psi & s\phi s\theta \\
s\phi c\psi + c\phi c\theta s\psi & -s\phi s\psi + c\phi c\theta c\psi & -c\phi s\theta \\
s\theta s\psi & s\theta c\psi & c\theta
\end{bmatrix}$$

Where: c = cos, s = sin

**Physical meaning**:
- **φ**: Longitude (rotation around vertical)
- **θ**: Latitude (tilt from vertical)
- **ψ**: Rotation about body axis

---

### Using the DCM

**Transform vectors between frames**:

If vector **v** is expressed in body frame:
$$\vec{v}_{\text{fixed}} = R \cdot \vec{v}_{\text{body}}$$

If vector **v** is expressed in fixed frame:
$$\vec{v}_{\text{body}} = R^T \cdot \vec{v}_{\text{fixed}}$$

**Example**: Aircraft
- Body frame: Forward (X), Right (Y), Down (Z)
- Fixed frame: North (X), East (Y), Down (Z)
- DCM transforms between them

---

## Angular Velocity from Euler Angle Rates

### The Challenge

**Given**: Euler angle rates $\dot{\phi}$, $\dot{\theta}$, $\dot{\psi}$

**Want**: Angular velocity vector **ω** in body frame

**Problem**: Rates are about different axes at different times!

### The Solution: Kinematic Equations

For **3-1-3 sequence** (ZXZ):

**Body frame components**:
$$\omega_x = \dot{\theta} \cos\psi + \dot{\phi} \sin\theta \sin\psi$$

$$\omega_y = -\dot{\theta} \sin\psi + \dot{\phi} \sin\theta \cos\psi$$

$$\omega_z = \dot{\psi} + \dot{\phi} \cos\theta$$

**Matrix form**:
$$\begin{bmatrix} \omega_x \\ \omega_y \\ \omega_z \end{bmatrix} = 
\begin{bmatrix}
\cos\psi & \sin\theta\sin\psi & 0 \\
-\sin\psi & \sin\theta\cos\psi & 0 \\
0 & \cos\theta & 1
\end{bmatrix}
\begin{bmatrix} \dot{\theta} \\ \dot{\phi} \\ \dot{\psi} \end{bmatrix}$$

**These are the kinematic differential equations!**

### Gimbal Lock

**Warning**: When θ = 0 or π:
- Two axes align (gimbal lock)
- Loss of one DOF
- Singularity in kinematic equations

**Solution**: Use quaternions for critical applications (spacecraft, robotics)

---

In [ ]:
# Calculate DCM symbolically for 3-1-3 Euler sequence

# Define symbolic variables
phi, theta, psi = symbols('phi theta psi', real=True)

# Individual rotation matrices
Rz_phi = Matrix([
    [cos(phi), -sin(phi), 0],
    [sin(phi),  cos(phi), 0],
    [0,         0,        1]
])

Rx_theta = Matrix([
    [1,    0,          0         ],
    [0,    cos(theta), -sin(theta)],
    [0,    sin(theta),  cos(theta)]
])

Rz_psi = Matrix([
    [cos(psi), -sin(psi), 0],
    [sin(psi),  cos(psi), 0],
    [0,         0,        1]
])

# Combined DCM (3-1-3 sequence: Z-X-Z)
R_313 = Rz_phi * Rx_theta * Rz_psi
R_313_simplified = simplify(R_313)

print("Direction Cosine Matrix (3-1-3 Euler Sequence)")
print("="*60)
print("Sequence: Z(φ) → X(θ) → Z(ψ)")
print()
print("DCM = R_z(φ) · R_x(θ) · R_z(ψ)")
print()
print("Simplified DCM:")
print()

# Display nicely
for i in range(3):
    row_str = "│ "
    for j in range(3):
        element = R_313_simplified[i, j]
        row_str += f"{latex(element):30s}  "
    row_str += "│"
    print(row_str)
    
print()
print("="*60)

# Verify orthogonality
RTR = simplify(R_313_simplified.T * R_313_simplified)
is_orthogonal = RTR == Matrix.eye(3)
print(f"\nVerification: R^T · R = I? {is_orthogonal} ")

# Numerical example: φ=30°, θ=45°, ψ=60°
phi_val = np.radians(30)
theta_val = np.radians(45)
psi_val = np.radians(60)

R_numeric = np.array(R_313_simplified.subs([
    (phi, phi_val),
    (theta, theta_val),
    (psi, psi_val)
]).evalf()).astype(float)

print(f"\nNumerical Example: φ={30}°, θ={45}°, ψ={60}°")
print("DCM =")
print(R_numeric)
print(f"\nDeterminant = {np.linalg.det(R_numeric):.6f} (should be 1.0) ")


## Application: Slider-Crank Mechanism

**The most important mechanism in mechanical engineering!**

**Found in**:
- Internal combustion engines (every car!)
- Compressors
- Pumps
- Manufacturing machinery

### Mechanism Description

**Components**:
1. **Crank**: Rotates with constant ω (driven by motor)
2. **Connecting rod**: Links crank to slider
3. **Slider**: Translates back and forth

**Goal**: Given crank rotation, find:
- Position of slider
- Velocity of slider
- Acceleration of slider

**This combines everything from Unit 3!**

---

### Kinematics Analysis

**Setup**:
- Crank length: r
- Rod length: L
- Crank angle: θ (from horizontal)
- Crank angular velocity: ω (constant)

**Position Analysis**:

Slider position x:
$$x = r\cos\theta + L\cos\phi$$

Where φ is rod angle (from horizontal).

**Constraint** (vertical position of connecting rod ends must match):
$$r\sin\theta = L\sin\phi$$

Solve for φ:
$$\sin\phi = \frac{r}{L}\sin\theta$$

For **r < L** (typical), φ is well-defined.

**Velocity**:

Differentiate position:
$$v = \frac{dx}{dt} = -r\omega\sin\theta - L\dot{\phi}\sin\phi$$

From constraint:
$$\dot{\phi} = \frac{r\omega}{L}\frac{\cos\theta}{\cos\phi}$$

Substitute:
$$v = -r\omega\left(\sin\theta + \frac{r}{L}\frac{\sin\theta\cos\theta}{\cos\phi}\right)$$

**Acceleration**:

Differentiate velocity:
$$a = \frac{dv}{dt} = -r\omega^2\cos\theta - L\ddot{\phi}\sin\phi - L\dot{\phi}^2\cos\phi$$

Calculate $\ddot{\phi}$ from differentiating $\dot{\phi}$ equation.

**Result**: Complex expressions, but computable!

---

### Numerical Example: Engine Slider-Crank

**Given**:
- Crank length: r = 0.05 m (50 mm)
- Rod length: L = 0.15 m (150 mm)
- RPM: 3000 rpm → ω = 314.16 rad/s
- Current angle: θ = 30°

**Find**: Position, velocity, acceleration of slider

---

In [ ]:
# Slider-Crank Mechanism Analysis

# Given parameters
r = 0.05  # m (crank radius)
L = 0.15  # m (rod length)
rpm = 3000  # rpm
omega = rpm * 2 * np.pi / 60  # rad/s
theta = np.radians(30)  # current angle

print("Slider-Crank Mechanism Analysis")
print("="*60)
print(f"Parameters:")
print(f"Crank radius: r = {r*1000:.1f} mm")
print(f"Rod length: L = {L*1000:.1f} mm")
print(f"Speed: {rpm} RPM = {omega:.2f} rad/s")
print(f"Current angle: θ = {np.degrees(theta):.1f}°")
print()

# Position analysis
sin_theta = np.sin(theta)
cos_theta = np.cos(theta)

# Rod angle from constraint
sin_phi = (r / L) * sin_theta
phi = np.arcsin(sin_phi)
cos_phi = np.cos(phi)

# Slider position
x = r * cos_theta + L * cos_phi

print(f"Position Analysis:")
print(f"Rod angle: φ = {np.degrees(phi):.2f}°")
print(f"Slider position: x = {x*1000:.2f} mm")
print()

# Velocity analysis
phi_dot = (r * omega / L) * (cos_theta / cos_phi)
v = -r * omega * sin_theta - L * phi_dot * sin_phi

print(f"Velocity Analysis:")
print(f"Rod angular velocity: φ̇ = {phi_dot:.2f} rad/s")
print(f"Slider velocity: v = {v:.2f} m/s = {v*1000:.1f} mm/s")
print()

# Acceleration analysis
# Differentiate phi_dot expression
phi_ddot = (r * omega**2 / L) * (
    -sin_theta / cos_phi
    + (r / L) * cos_theta * sin_theta * sin_phi / cos_phi**3
)

a = -r * omega**2 * cos_theta - L * phi_ddot * sin_phi - L * phi_dot**2 * cos_phi

print(f"Acceleration Analysis:")
print(f"Rod angular acceleration: φ̈ = {phi_ddot:.2f} rad/s²")
print(f"Slider acceleration: a = {a:.2f} m/s² = {a/9.81:.2f} g")
print("="*60)
print()

# Animation of mechanism over full cycle
print("Generating mechanism animation...")
anim = animate_slider_crank(
    r_crank=r,
    l_rod=L,
    omega=omega/10,  # Slow down for visualization
    duration=4.0,
    fps=30
)
print("Animation complete!")
print()
print("KEY OBSERVATIONS:")
print("1. Slider motion is NOT sinusoidal (due to finite rod length)")
print("2. Acceleration is highest at extremes (TDC/BDC)")
print("3. Rod angle φ lags behind crank angle θ")
print("4. This is the heart of every piston engine!")


### Engineering Insights

**Design trade-offs**:

1. **Rod length ratio (L/r)**:
   - Typical: L/r = 3 to 4
   - Larger ratio → more sinusoidal motion
   - Smaller ratio → more compact but higher side forces

2. **Speed effects**:
   - Higher RPM → higher accelerations
   - a ∝ ω² (quadratic!)
   - 3000 RPM → very high accelerations (100s of g)

3. **Forces** (preview of Unit 5):
   - Must account for inertial forces: F = m·a
   - At high speeds, inertial forces dominate
   - Balancing critical for smooth operation

4. **Real engines**:
   - Multi-cylinder: phase offset for smoothness
   - Counterweights: balance inertial forces
   - Bearings: handle these extreme accelerations

**This mechanism showcases ALL Unit 3 concepts**:
- Position, velocity, acceleration
- Rotating reference frames (rod frame)
- Constraints (kinematic relationships)
- 2D rigid body motion

---

## Practice Problems

Apply Euler angles and mechanism analysis!

---

In [ ]:
# Workspace for Practice Problem 1
# Write your code below
# 
# 
# 
# 
# 
# 
# 
# 
# 
# 


### Practice Problem 1: Spacecraft Attitude

A spacecraft has Euler angles (3-1-3 sequence):
- φ = 30°, θ = 60°, ψ = 45°
- Rates: φ̇ = 0.1 rad/s, θ̇ = 0.05 rad/s, ψ̇ = 0.2 rad/s

Find:
1. DCM from body to inertial frame
2. Angular velocity vector in body frame

**Hint**: Use kinematic equations for ω components.

---

In [ ]:
# Workspace for Practice Problem 2
# Write your code below
# 
# 
# 
# 
# 
# 
# 
# 
# 
# 


### Solution

*Solution available in teacher version. Attempt the problem first, then check with instructor.*

---

### Practice Problem 2: Mini Slider-Crank

A model engine has r = 20 mm, L = 60 mm, running at 6000 RPM.

At θ = 90° (crank vertical), find:
1. Slider velocity
2. Slider acceleration
3. Compare to gravitational acceleration

---

In [ ]:
# Workspace for Practice Problem 3
# Write your code below
# 
# 
# 
# 
# 
# 
# 
# 
# 
# 


### Solution

*Solution available in teacher version. Attempt the problem first, then check with instructor.*

---

## Unit 3 Complete Summary

### Five Lectures, Complete Kinematics

**L1: Fixed Frame - Velocity**
$$\boxed{\vec{v}_P = \vec{v}_{\text{center}} + \vec{\omega} \times \vec{r}}$$

**L2: Fixed Frame - Acceleration**
$$\boxed{\vec{a}_P = \vec{a}_{\text{center}} + \vec{\alpha} \times \vec{r} + \vec{\omega} \times (\vec{\omega} \times \vec{r})}$$

**L3: Transport Theorem**
$$\boxed{\left(\frac{d\vec{Q}}{dt}\right)_{\text{fixed}} = \left(\frac{d\vec{Q}}{dt}\right)_{\text{rotating}} + \vec{\Omega} \times \vec{Q}}$$

**L4: Coriolis Acceleration**
$$\boxed{\vec{a}_{\text{fixed}} = \vec{a}_{\text{rel}} + 2\vec{\Omega} \times \vec{v}_{\text{rel}} + \vec{\alpha} \times \vec{r} + \vec{\Omega} \times (\vec{\Omega} \times \vec{r})}$$

**L5: 3D Rotations**
$$\boxed{R = R_z(\phi) \cdot R_x(\theta) \cdot R_z(\psi)}$$
$$\boxed{\vec{\omega} = f(\dot{\phi}, \dot{\theta}, \dot{\psi})}$$

### Key Concepts Mastered

1. **Rigid body definition**: Fixed internal distances
2. **DOF**: 3 (2D) or 6 (3D) for rigid body
3. **Velocity decomposition**: Translation + rotation
4. **Acceleration components**: Tangential + normal (centripetal)
5. **Reference frames**: Fixed vs rotating perspectives
6. **Coriolis effect**: Most important rotating frame phenomenon
7. **Euler angles**: 3D orientation parameterization
8. **Mechanisms**: Practical application (slider-crank)

### Applications Covered

- Rolling wheels
- Rotating disks and rods
- Hurricanes and weather
- Foucault pendulum
- Artillery deflection
- Aircraft navigation
- Spacecraft attitude
- Engine mechanisms

### Mathematics Toolkit

- Vector algebra (dot, cross products)
- Time derivatives in different frames
- Rotation matrices and DCM
- Kinematic differential equations
- Symbolic computation (SymPy)
- Numerical integration (for mechanisms)

---

## Looking Ahead: From Kinematics to Kinetics

### Unit 3 vs Units 4-5

**Unit 3 (Kinematics)**: **HOW** things move
- Given: Forces, constraints, initial conditions
- Find: Position, velocity, acceleration
- No mass, no forces considered

**Units 4-5 (Kinetics)**: **WHY** things move
- Given: Masses, forces, torques
- Find: Equations of motion, trajectories
- Newton's laws (F = ma), Euler's equations (M = Iα)

### The Bridge

**Kinematics provides**:
- Velocity and acceleration expressions
- Kinematic constraints
- Reference frame transformations

**Kinetics adds**:
- Forces and torques
- Mass and inertia
- Newton's second law
- → **Equations of motion**

### Unit 4 Preview: Particle Kinetics

**Next unit**:
- Pulleys and mechanical advantage
- Constrained systems (gantry robots, 3D printers)
- Work-energy methods
- Numerical simulations

### Unit 5 Preview: Rigid Body Kinetics

**After Unit 4**:
- Moments of inertia
- Euler's equations for rotation
- 1-DOF and 2-DOF systems
- Gyroscopic effects

**The complete toolkit**: Kinematics (Unit 3) + Kinetics (Units 4-5) = Full dynamics!

---

## Homework

1. **Conceptual**: Explain the difference between "Euler angles" and "Tait-Bryan angles". Give examples of where each is used.

2. **DCM Calculation**: For Euler angles φ=45°, θ=30°, ψ=60° (3-1-3 sequence):
   - Construct DCM numerically
   - Verify orthogonality (R^T·R = I)
   - Transform vector v = (1, 0, 0) from body to inertial frame

3. **Slider-Crank**: Design a slider-crank with:
   - Stroke = 100 mm (total travel)
   - L/r ratio = 3.5
   - Find r and L
   - Calculate max velocity and acceleration at 2000 RPM

4. **Unit 3 Integration**: A disk of radius 0.3 m rotates at Ω = 5 rad/s. An ant walks radially outward at 0.1 m/s starting at r = 0.1 m. Using all Unit 3 concepts:
   - Position vs time r(t)
   - Velocity in fixed frame
   - All acceleration components
   - Path shape in fixed frame

5. **Coding Challenge**: Implement a 3D visualization of Euler angle rotations. Show how the body frame axes transform as you vary φ, θ, ψ independently.

**Submit**: Comprehensive Jupyter notebook demonstrating Unit 3 mastery!

---

## Congratulations!

**You've completed Unit 3: Rigid Body Kinematics!**

You now understand:
- How rigid bodies move in 2D and 3D
- Fixed and rotating reference frames
- The mysterious Coriolis effect
- 3D rotations and Euler angles
- Real mechanisms and applications

**Total**: 5 hours of lecture content, foundations for all advanced dynamics!

**Next stop**: Particle Kinetics (Unit 4) - where forces enter the picture! 

---

**End of Unit 3, Lecture 5** 
